# Level probe: which comparison can you make?

A way to try the binding idea by ear before building an experiment on it.

Every trial is two sounds: the four-tone figure from the jitter task, five repetitions, at one
step. In one of the two sounds the red tone (2241 Hz) is a little louder on the last repetition.
You say which.

Loudness is scrambled on purpose (the rove), so that only one comparison gives the answer:

| | scrambled | what still works | needs |
|---|---|---|---|
| **A** | every repetition, all four tones together | the red tone against the other three at the same moment: the colour of the last chord | the tones heard together |
| **B** | the red tone's whole sequence, and the other three on the last repetition | the red tone against its own earlier repetitions | the red tone heard on its own |

The level information is physically the same at every step. If the step changes how hard A and
B are, it is because it changed what you hear as one thing.

**If the idea works, A gets harder and B gets easier as the step grows.**

| part | |
|---|---|
| 1 | the idea in one figure |
| 2 | without the rove, for comparison |
| 3 | trial type A at each step |
| 4 | trial type B at each step |
| 5 | what a result would look like |
| 6 | test yourself |
| 7 | your results |
| 8 | build your own |
| 9 | why the rove is 10 dB |

Use headphones. In the listening sections each row is one random draw and the left sound is the
one with the louder red tone; run a cell again for new draws. You can replay as often as you
like here, whereas in an experiment each sound plays once.

Run the setup cell first, then use the outline (☰, left) to move around.

In [ ]:
#@title setup (run this first)
import sys, subprocess, importlib.util, io, base64, wave, math, csv, time
from pathlib import Path
REF = 'bab4e44ee9412dbf800314e9ac2cf2fc175347b4'
REPO = 'https://github.com/MeysamAmirsardari/SeqSFG_task.git'
try:
    import tshear.levelprobe
except Exception:
    if bool(importlib.util.find_spec('google') and importlib.util.find_spec('google.colab')):
        ROOT = Path('/content') / ('seqsfg-' + REF[:12])
        if not ROOT.exists():
            subprocess.run(['git', 'clone', '--no-checkout', REPO, str(ROOT)], check=True)
            subprocess.run(['git', '-C', str(ROOT), 'checkout', '--detach', REF], check=True)
    else:
        ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / 'tshear/levelprobe.py').exists()), None)
    for _m in [k for k in list(sys.modules) if k.split('.')[0] in ('tshear', 'tcoh')]:
        del sys.modules[_m]
    sys.path.insert(0, str(ROOT))

get_ipython().run_line_magic('matplotlib', 'inline')
import numpy as np, matplotlib
matplotlib.rcParams.update({'figure.dpi': 120, 'font.size': 8,
                            'axes.spines.top': False, 'axes.spines.right': False})
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from IPython.display import HTML, display
import ipywidgets as W

import tshear.config as TC
from tshear import levelprobe as L

CFG = TC.Config()
SR = CFG.sample_rate
T = CFG.target_index
OTHERS = [j for j in range(CFG.n_tones) if j != T]
STEPS = list(L.STEPS_MS)
LOUDER = [3.0, 6.0, 9.0, 12.0]
RED, GREY = '#c0392b', '#5d6d7e'
COL = {'A': '#1a5276', 'B': '#b9770e'}
MIN_ANSWER_S = 0.4          # ignore an answer arriving this soon after a new trial (double clicks)


# ---- audio --------------------------------------------------------------------------
def _wav(x, peak):
    d = (np.clip(np.asarray(x, float) / max(peak, 1e-12) * 0.85, -1, 1) * 32767).astype('<i2')
    b = io.BytesIO()
    with wave.open(b, 'wb') as w:
        w.setnchannels(1); w.setsampwidth(2); w.setframerate(SR); w.writeframes(d.tobytes())
    return base64.b64encode(b.getvalue()).decode()

def _player(x, peak):
    return ('<audio controls preload="none" style="height:32px" '
            f'src="data:audio/wav;base64,{_wav(x, peak)}"></audio>')

def pairs_html(rows, heads=('louder at the end', 'not louder'), note=''):
    # Both sounds of a row share one scale factor, so their level relation is the real one.
    labelled = any(lab for lab, _, _ in rows)
    th = 'style="text-align:left;padding:4px 10px"'
    h = ['<table style="border-collapse:collapse;font:13px system-ui;max-width:780px">', '<tr>']
    if labelled:
        h.append(f'<th {th}>louder by</th>')
    h += [f'<th {th}>{heads[0]}</th><th {th}>{heads[1]}</th></tr>']
    for lab, a, b in rows:
        pk = max(float(np.abs(a).max()), float(np.abs(b).max()))
        h.append('<tr style="border-top:1px solid #e3e3e3">')
        if labelled:
            h.append(f'<td style="padding:6px 10px;color:{RED};font-weight:600">{lab}</td>')
        h.append(f'<td style="padding:6px 10px">{_player(a, pk)}</td>'
                 f'<td style="padding:6px 10px">{_player(b, pk)}</td></tr>')
    h.append('</table>')
    if note:
        h.append(f'<div style="font:12px system-ui;color:#666;margin-top:6px">{note}</div>')
    return ''.join(h)

def wrong_cue(task, db, rove=L.ROVE_DB, n=6000):
    # % correct for a listener who uses ONLY the comparison this trial type scrambles
    return 100 * L.p_correct(CFG, task, db, rove, w_timbre=0.0 if task == 'A' else 1.0, n=n)

WRONG = {(t, d): wrong_cue(t, d) for t in 'AB' for d in LOUDER}

def listen(task, step, rove=L.ROVE_DB):
    rows = []
    for d in LOUDER:
        tr = L.make_trial(task, step, d, np.random.default_rng(), rove_db=rove, cfg=CFG,
                          target_position=1)
        rows.append((f'{d:g} dB', L.render(CFG, tr.first, tr.phases),
                     L.render(CFG, tr.second, tr.phases)))
    if task in ('A', 'B'):
        note = ('Using only the scrambled comparison you would score '
                + ', '.join(f'{WRONG[(task, d)]:.0f}%' for d in LOUDER)
                + ' on these four rows. One random draw per row; run again for new ones.')
    else:
        note = 'No rove, so both comparisons work. One random draw per row.'
    display(HTML(pairs_html(rows, note=note)))


# ---- plots --------------------------------------------------------------------------
def token_plot(step, cfg=CFG):
    on = L.onsets(cfg, step) - cfg.lead_ms
    t = cfg.target_index
    fig, ax = plt.subplots(figsize=(7.4, 1.9))
    for rep in range(cfg.n_repeats):
        for k in range(cfg.n_tones):
            ax.add_patch(Rectangle((on[rep, k], k - .34), cfg.tone_ms, .68,
                                   color=RED if k == t else GREY, alpha=.95 if k == t else .72))
    ax.plot(on[-1, t] + cfg.tone_ms / 2, t + .5, marker='^', color=RED, ms=5, clip_on=False)
    ax.text(1.0, 1.02, '▲ louder on the last repetition, in one of the two sounds',
            transform=ax.transAxes, ha='right', va='bottom', fontsize=6, color=RED)
    span = (cfg.n_tones - 1) * step
    ax.set_xlim(-30, (cfg.n_repeats - 1) * cfg.period_ms + span + cfg.tone_ms + 40)
    ax.set_ylim(-.7, cfg.n_tones - .25)
    ax.set_yticks(range(cfg.n_tones))
    ax.set_yticklabels([f'{f:.0f}' for f in cfg.freqs_hz])
    ax.set_xlabel('ms'); ax.grid(axis='x', alpha=.15)
    plt.show()

def wilson(k, n, z=1.96):
    if n == 0:
        return 0.0, 1.0
    p = k / n; d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return c - h, c + h

print(f'ready  ·  {CFG.n_tones} tones '
      + ' + '.join(f'{f:.0f}' for f in CFG.freqs_hz)
      + f' Hz at {CFG.rate_hz:g} Hz, {CFG.n_repeats} repetitions  ·  rove ±{L.ROVE_DB:g} dB')

---
# 1 · The idea in one figure

Level of each tone across the five repetitions of one sound. Red is the target, grey the other
three. Left: the sound where red is louder at the end. Right: the other sound.

In [ ]:
#@title levels in the two sounds
def _find(task, want, db=6.0):
    for seed in range(5000):
        tr = L.make_trial(task, 25.0, db, np.random.default_rng(seed), target_position=1)
        if want(tr):
            return tr
    raise RuntimeError('no illustrative draw found')

def _misleads_A(tr):   # the red tone alone is LOUDER at the end in the wrong sound
    return tr.second.gains_db[-1, T] > tr.first.gains_db[-1, T] + 2
def _misleads_B(tr):   # red against grey points to the wrong sound
    tim = lambda g: g[-1, T] - g[-1, OTHERS].mean()
    return tim(tr.second.gains_db) > tim(tr.first.gains_db) + 2

draws = [('no rove', L.make_trial('none', 25.0, 6.0, np.random.default_rng(0), target_position=1)),
         ('A', _find('A', _misleads_A)), ('B', _find('B', _misleads_B))]
reps = np.arange(1, CFG.n_repeats + 1)
WORKS = {'no rove': ('both comparisons work', '#333'),
         'A': ('red against grey, same moment: works', COL['A']),
         'B': ('red against its own past: works', COL['B'])}
FOOLS = {'A': 'red on its own is louder here,\nin the sound that is NOT louder',
         'B': 'red against grey is bigger here,\nin the sound that is NOT louder'}
fig, axs = plt.subplots(3, 2, figsize=(8.4, 6.8), sharex=True, sharey=True)
for r, (name, tr) in enumerate(draws):
    for c, iv in enumerate((tr.first, tr.second)):
        ax = axs[r, c]
        g = iv.gains_db
        for j in OTHERS:      # wide and pale, so a grey line hidden under the red one still shows
            ax.plot(reps, g[:, j], '-', color=GREY, lw=6, alpha=.22, solid_capstyle='round')
            ax.plot(reps, g[:, j], 'o', color=GREY, ms=3.5, alpha=.85)
        ax.plot(reps, g[:, T], '-o', color=RED, lw=1.8, ms=5, zorder=3)
        ax.axvspan(4.6, 5.4, color='#f2f3f4', zorder=0)
        ax.set_ylim(-12, 16); ax.grid(alpha=.2); ax.set_xticks(reps)
        if c == 0:
            last = g[-1, T]
            ref = g[-1, OTHERS].mean() if name == 'A' else g[:-1, T].mean()
            if name == 'B':
                ax.plot([1, 5.3], [ref, ref], ':', color=COL['B'], lw=1)
            ax.annotate('', (5.3, last), (5.3, ref),
                        arrowprops=dict(arrowstyle='->', color=WORKS[name][1], lw=1.4,
                                        shrinkA=0, shrinkB=2))
            ax.text(.98, .96, WORKS[name][0], transform=ax.transAxes, ha='right', va='top',
                    fontsize=7.5, color=WORKS[name][1])
        elif name in FOOLS:
            ax.text(.98, .96, FOOLS[name], transform=ax.transAxes, ha='right', va='top',
                    fontsize=7, color='#777')
        if r == 0:
            ax.set_title(('louder at the end', 'not louder')[c], fontsize=9)
        if c == 0:
            ax.set_ylabel(f'{name}\nlevel (dB)', fontsize=9)
        if r == 2:
            ax.set_xlabel('repetition')
axs[0, 0].text(.02, .96, 'red: the target\ngrey: the other three', transform=axs[0, 0].transAxes,
               ha='left', va='top', fontsize=7, color='#555')
plt.tight_layout(); plt.show()

**No rove.** Red rises against the others and against its own past, so either comparison gives
it away, whatever the grouping.

**A.** Everything jumps together on every repetition. Red on its own is useless; in this draw it
is even louder in the wrong sound. Only red against grey in the last column works.

**B.** Red's whole line is shifted, and the grey tones scatter at the end. Red against grey is
useless; in this draw it points to the wrong sound. Only red against its own past works.

That is the whole link between level and binding: which comparison you can make depends on
whether the red tone is heard with the others or on its own.

---
# 2 · Without the rove

The louder tone is audible however the tones are grouped: the whole thing gets louder and its
colour changes. This is why the rove is needed. Without it the test says nothing about binding.

## no rove · 0 ms

In [ ]:
#@title plot and audio
token_plot(0.0)
listen('none', 0.0)

## no rove · 75 ms

In [ ]:
#@title plot and audio
token_plot(75.0)
listen('none', 75.0)

---
# 3 · Trial type A: needs the tones heard together

Loudness jumps on every repetition, all four tones together, so the red tone's own loudness is
useless. What still works is red against the other three at the same moment, which you hear as
the colour of the last chord changing.

## A · 0 ms

All four start together.

In [ ]:
#@title plot and audio
token_plot(0.0)
listen('A', 0.0)

## A · 10 ms

A slightly smeared chord.

In [ ]:
#@title plot and audio
token_plot(10.0)
listen('A', 10.0)

## A · 25 ms

Around where onset differences start to pull sounds apart.

In [ ]:
#@title plot and audio
token_plot(25.0)
listen('A', 25.0)

## A · 45 ms

An arpeggio. Neighbours still overlap by 15 ms.

In [ ]:
#@title plot and audio
token_plot(45.0)
listen('A', 45.0)

## A · 75 ms

Four separate notes, no overlap.

In [ ]:
#@title plot and audio
token_plot(75.0)
listen('A', 75.0)

---
# 4 · Trial type B: needs the red tone heard on its own

The red tone's whole sequence sits at a random level, and the other three jump at random on the
last repetition, so the colour of the last chord changes in both sounds. What still works is red
against its own earlier repetitions.

## B · 0 ms

All four start together.

In [ ]:
#@title plot and audio
token_plot(0.0)
listen('B', 0.0)

## B · 10 ms

A slightly smeared chord.

In [ ]:
#@title plot and audio
token_plot(10.0)
listen('B', 10.0)

## B · 25 ms

Around where onset differences start to pull sounds apart.

In [ ]:
#@title plot and audio
token_plot(25.0)
listen('B', 25.0)

## B · 45 ms

An arpeggio. Neighbours still overlap by 15 ms.

In [ ]:
#@title plot and audio
token_plot(45.0)
listen('B', 45.0)

## B · 75 ms

Four separate notes, no overlap.

In [ ]:
#@title plot and audio
token_plot(75.0)
listen('B', 75.0)

---
# 5 · What a result would look like

Two imaginary listeners at 4 dB, computed on the real trials. Left: the tones fuse at small
steps and come apart at large ones, so the listener can only use the comparison the grouping
allows. Right: grouping never changes, every comparison just gets noisier as the tones spread.
Dotted: using only the scrambled comparison.

In [ ]:
#@title two imaginary listeners
grid = np.linspace(0, 75, 16)
db = 4.0
def _fuse(s):
    return 1 / (1 + np.exp((s - 25) / 7))
curves = {}
for who in ('fuse', 'harder'):
    for task in 'AB':
        ys = []
        for s in grid:
            if who == 'fuse':
                w, k = _fuse(s), 1.0
            else:
                w, k = (1.0 if task == 'A' else 0.0), 1 + s / 40
            ys.append(100 * L.p_correct(CFG, task, db, L.ROVE_DB, w_timbre=w,
                                        noise_timbre_db=2.5 * k, noise_own_db=1.5 * k,
                                        n=8000, seed=1))
        curves[(who, task)] = ys
fig, axs = plt.subplots(1, 2, figsize=(9.6, 3.3), sharey=True)
for ax, who, title in zip(axs, ('fuse', 'harder'),
                          ('tones fuse, then come apart', 'nothing fuses, it just gets harder')):
    for task in 'AB':
        ax.plot(grid, curves[(who, task)], '-', color=COL[task], lw=2.2, label=task)
        ax.axhline(wrong_cue(task, db), color=COL[task], ls=':', lw=1)
    ax.text(75, wrong_cue('A', db) - 1.5, 'scrambled comparison only', ha='right', va='top',
            fontsize=7, color='#777')
    ax.axhline(50, color='#999', lw=.8)
    ax.set_ylim(40, 102); ax.set_xticks(STEPS); ax.grid(alpha=.25)
    ax.set_xlabel('step (ms)'); ax.set_title(title, fontsize=9)
axs[0].set_ylabel('% correct'); axs[0].legend(frameon=False, fontsize=8, loc='lower left')
plt.tight_layout(); plt.show()

Left is the pattern the idea predicts: the two lines cross. Right is the main alternative: both
fall together. The step does not change the level information at all; only the listener can.

---
# 6 · Test yourself

Choose the settings and press start. Play both sounds, then answer: which one had the red tone
louder at the end? *A and B mixed* with *all steps mixed* is the real design. A trial takes about
ten seconds, so 100 trials (about ten per point) is a quarter of an hour.

In [ ]:
#@title test
RESULTS = globals().get('RESULTS', [])
_wide = {'description_width': 'initial'}
w_type = W.Dropdown(options=[('A and B mixed', 'AB'), ('A only', 'A'), ('B only', 'B')],
                    value='AB', description='trial type', style=_wide)
w_step = W.Dropdown(options=[('all steps mixed', 'all')] + [(f'{s:g} ms', s) for s in STEPS],
                    value='all', description='step', style=_wide)
w_db = W.BoundedFloatText(value=4.0, min=0.5, max=12.0, step=0.5,
                          description='louder by (dB)', style=_wide)
w_rove = W.BoundedFloatText(value=L.ROVE_DB, min=0.0, max=12.0, step=1.0,
                            description='rove ± (dB)', style=_wide)
w_n = W.IntSlider(value=40, min=10, max=200, step=10, description='trials', style=_wide)
b_start = W.Button(description='start', button_style='primary')
b1 = W.Button(description='1', layout=W.Layout(width='90px'))
b2 = W.Button(description='2', layout=W.Layout(width='90px'))
stage, status = W.Output(), W.HTML()
_S = {'trial': None}

def _show():
    task, step = _S['order'][_S['i']]
    tr = L.make_trial(task, step, _S['db'], _S['rng'], rove_db=_S['rove'], cfg=CFG)
    _S['trial'] = tr
    x1, x2 = L.render(CFG, tr.first, tr.phases), L.render(CFG, tr.second, tr.phases)
    with stage:
        stage.clear_output(wait=True)
        display(HTML(pairs_html([('', x1, x2)], heads=('1', '2'))))
    status.value = (f"trial {_S['i'] + 1} of {len(_S['order'])}" + _S['last'])
    _S['shown'] = time.time()
    b1.disabled = b2.disabled = False

def _start(_=None):
    tasks = ['A', 'B'] if w_type.value == 'AB' else [w_type.value]
    steps = STEPS if w_step.value == 'all' else [w_step.value]
    rng = np.random.default_rng()
    cells_ = [(t, s) for t in tasks for s in steps]
    order = []
    while len(order) < w_n.value:            # balanced: every cell once before any repeats
        blk = list(cells_); rng.shuffle(blk); order += blk
    _S.update(order=order[:w_n.value], i=0, rng=rng, db=float(w_db.value),
              rove=float(w_rove.value), ok=0, last='', run=time.strftime('%Y-%m-%d %H:%M:%S'))
    _show()

def _answer(k):
    tr = _S.get('trial')
    if tr is None or time.time() - _S.get('shown', 0) < MIN_ANSWER_S:
        return
    ok = int(k == tr.target_position)
    _S['ok'] += ok
    RESULTS.append(dict(run=_S['run'], task=tr.task, step_ms=tr.step_ms, louder_db=tr.delta_db,
                        rove_db=tr.rove_db, answer=k, target=tr.target_position, correct=ok))
    _S['last'] = ('&nbsp;&nbsp;<span style="color:#1e8449">last one: correct</span>' if ok else
                  f'&nbsp;&nbsp;<span style="color:{RED}">last one: it was {tr.target_position}</span>')
    _S['i'] += 1
    if _S['i'] >= len(_S['order']):
        _S['trial'] = None
        b1.disabled = b2.disabled = True
        with stage:
            stage.clear_output()
        status.value = (f"done: {_S['ok']} of {len(_S['order'])} correct. "
                        'Run the next cell for the results.')
    else:
        _show()

b_start.on_click(_start)
b1.on_click(lambda _: _answer(1))
b2.on_click(lambda _: _answer(2))
b1.disabled = b2.disabled = True
display(W.VBox([W.HBox([w_type, w_step]), W.HBox([w_db, w_rove]), W.HBox([w_n, b_start]),
                stage, W.HBox([b1, b2]), status]))

---
# 7 · Your results

If the idea works: A (blue) high at 0 ms and falling, B (amber) low at 0 ms and rising. Near the
dotted line means you could not use the intended comparison. Bars are 95% intervals; with ten
trials a point, only a large effect shows.

If B is easy even at 0 ms, these four tones were not fused even when they start together. That
is a finding about this chord, not a failure of the method.

In [ ]:
#@title results
if not RESULTS:
    print('no trials yet: run the test above first')
else:
    groups = sorted({(r['louder_db'], r['rove_db']) for r in RESULTS})
    fig, axs = plt.subplots(1, len(groups), figsize=(4.6 * len(groups), 3.4), squeeze=False)
    print(f"{'type':>4} {'step':>7} {'n':>4} {'correct':>8}   louder by, rove")
    for ax, (db, rv) in zip(axs[0], groups):
        for task in 'AB':
            pts = []
            for s in STEPS:
                c = [r['correct'] for r in RESULTS if r['task'] == task and r['step_ms'] == s
                     and r['louder_db'] == db and r['rove_db'] == rv]
                if c:
                    lo, hi = wilson(sum(c), len(c))
                    pts.append((s, 100 * np.mean(c), 100 * lo, 100 * hi, len(c)))
                    print(f'{task:>4} {s:5g} ms {len(c):4d} {100 * np.mean(c):7.0f}%   '
                          f'{db:g} dB, ±{rv:g} dB')
            if pts:
                x = np.array([p[0] for p in pts]) + (-1.2 if task == 'A' else 1.2)
                y = [p[1] for p in pts]
                ax.errorbar(x, y, yerr=[[p[1] - p[2] for p in pts], [p[3] - p[1] for p in pts]],
                            fmt='-o', color=COL[task], lw=2, ms=6, capsize=2, label=task)
                if rv > 0:
                    ax.axhline(wrong_cue(task, db, rv), color=COL[task], ls=':', lw=1)
        ax.axhline(50, color='#999', lw=.8)
        ax.set_ylim(30, 104); ax.set_xticks(STEPS); ax.grid(alpha=.25)
        ax.set_xlabel('step (ms)'); ax.set_ylabel('% correct')
        ax.set_title(f'louder by {db:g} dB, rove ±{rv:g} dB', fontsize=9)
        ax.legend(frameon=False, fontsize=8, loc='lower left')
    plt.tight_layout(); plt.show()
    out = Path('level_probe_results.csv')
    with out.open('w', newline='') as f:
        wr = csv.DictWriter(f, fieldnames=list(RESULTS[0])); wr.writeheader(); wr.writerows(RESULTS)
    print(f'\n{len(RESULTS)} trials saved to {out.resolve()}')

---
# 8 · Build your own

Set the numbers and run. Tones may overlap. Closer spacing (fewer semitones) should fuse more
strongly than the default.

In [ ]:
#@title build a trial
trial_type  = 'A'   #@param ["A", "B", "none"]
step_ms     = 25    #@param {type:"number"}
louder_db   = 6     #@param {type:"number"}
rove_db     = 10    #@param {type:"number"}
tone_ms     = 60    #@param {type:"number"}
rate_hz     = 3.0   #@param {type:"number"}
n_repeats   = 5     #@param {type:"integer"}
n_tones     = 4     #@param {type:"integer"}
lowest_hz   = 683   #@param {type:"number"}
spacing_st  = 10.2  #@param {type:"number"}
target_tone = 3     #@param {type:"integer"}

n_tones = max(int(n_tones), 2)
freqs = tuple(lowest_hz * 2 ** (spacing_st * k / 12) for k in range(n_tones))
cfg = TC.Config(freqs_hz=freqs, rate_hz=rate_hz, tone_ms=tone_ms, n_repeats=max(int(n_repeats), 2),
                target_index=min(max(int(target_tone), 1), n_tones) - 1)
total = L.interval_ms(cfg, max(step_ms, 0.0))
tr = L.make_trial(trial_type, step_ms, louder_db, np.random.default_rng(), rove_db=rove_db,
                  cfg=cfg, target_position=1)
period = cfg.period_ms
span = (n_tones - 1) * step_ms
print(' + '.join(f'{f:.0f}' for f in freqs) + ' Hz')
print(f'period {period:.0f} ms, figure spans {span:.0f} ms, '
      + ('tones overlap' if 0 < step_ms < tone_ms else 'tones do not overlap' if step_ms else 'all synchronous')
      + (f', and the figure is longer than the period' if span + tone_ms > period else ''))
if trial_type in ('A', 'B'):
    w = 0.0 if trial_type == 'A' else 1.0
    print(f'using only the scrambled comparison: '
          f'{100 * L.p_correct(cfg, trial_type, louder_db, rove_db, w_timbre=w, n=6000):.0f}% correct')
token_plot(step_ms, cfg)
display(HTML(pairs_html([(f'{louder_db:g} dB', L.render(cfg, tr.first, tr.phases, total),
                          L.render(cfg, tr.second, tr.phases, total))])))

---
# 9 · Why the rove is 10 dB

No rove hides the scrambled comparison completely. These curves are the threshold of a listener
who uses only that comparison: the increase at which they reach 79% correct. A real threshold
well below the curve means the intended comparison was used. At 5 dB the curve is already near
4 dB, about what a four-tone timbre comparison gives, so the test could not tell the two apart.
Above about 11 dB the waveform clips at the level an experiment would use.

In [ ]:
#@title the rove size
roves = np.arange(0, 14.5, 1.0)
fig, ax = plt.subplots(figsize=(6.2, 3.3))
for task in 'AB':
    ax.plot(roves, [L.ceiling(CFG, task, r, n=6000) for r in roves], '-o', color=COL[task],
            lw=2, ms=4, label=f'{task}: scrambled comparison only')
clip = next(r for r in roves if L.worst_case_peak(CFG, r, 12.0) >= 1.0)
ax.axvspan(clip - .5, roves[-1] + .5, color='#fdedec', zorder=0)
ax.text(clip, 1.0, ' clips', color=RED, fontsize=8)
ax.axvline(L.ROVE_DB, color='#999', ls='--', lw=1)
ax.set_xlim(roves[0] - .5, roves[-1] + .5)
ax.set_xlabel('rove ± (dB)'); ax.set_ylabel('threshold (dB)'); ax.grid(alpha=.25)
ax.legend(frameon=False, fontsize=8, loc='upper left')
plt.tight_layout(); plt.show()